Signor 3.0

In [3]:
targetGene = 'GBA'

In [3]:
import pandas as pd
final_df = pd.DataFrame(columns=['Database','TF','TargetGene','Library/PMID'])

In [ ]:
import http.client
import pandas as pd
from io import StringIO

# ids of Fatty Acid Biosynthesis

ids = ['O14757']

final_df = pd.DataFrame(columns=['Database','TF','TargetGene','Library/PMID'])

for uid in ids:

    # Establish connection
    conn = http.client.HTTPSConnection("signor.uniroma2.it")

    # Define headers
    headersList = {
        "Accept": "*/*",
        "User-Agent": "Thunder Client (https://www.thunderclient.com)"
    }

    # Empty payload
    payload = ""

    # Make the request
    conn.request("GET", f"/getData.php?id={uid}&organism=9606", payload, headersList)

    # Get the response
    response = conn.getresponse()
    result = response.read()

    # Decode the response as a string
    decoded_result = result.decode("utf-8")

    # Convert the string data into a pandas DataFrame
    data = StringIO(decoded_result)
    df1 = pd.read_csv(data, sep='\t',header = None)

    if(len(df1) <= 1):
        continue

    print(df1)

    mask = (df1[9] == 'transcriptional regulation')

    for i in range(0,len(df1[mask])):
        new_row = {'Database' : 'Signor', 'TF' : list(df1[mask][0])[i],'TargetGene':list(df1[mask][4])[i],'Library/PMID':list(df1[mask][21])[0]}
        final_df = pd.concat([final_df, pd.DataFrame([new_row])],ignore_index = True)


                                                   0         1             2   \
0                                               CHEK1   protein        O14757   
1                                               CHEK1   protein        O14757   
2                                               CHEK1   protein        O14757   
3   3-(carbamoylamino)-5-(3-fluorophenyl)-N-[(3S)-...  chemical  CHEBI:131156   
4                                               CHEK1   protein        O14757   
..                                                ...       ...           ...   
79                                              CHEK1   protein        O14757   
80                                               CDK1   protein        P06493   
81                                               CDK2   protein        P24941   
82                                              CHEK1   protein        O14757   
83                                              CHEK1   protein        O14757   

         3      4        5 

In [2]:
final_df

,Database,TF,TargetGene,Library/PMID
0,Signor,MEK1/2,CHEK1,28138032



ChEA3

In [1]:
#### It returns all the TFs and a mapping to which query gene they belong to under the attribute Overlapping_Gene####

import requests
import json

# List of genes
targetGene = 'PLA2G2A'
genes = [targetGene,"SCD"]

# URL of the ChEA3 API endpoint
url = "https://maayanlab.cloud/chea3/api/enrich/"

# Payload to send with the POST request
payload = {
    "query_name": "myQuery",
    "gene_set": genes
}

# POST to ChEA3 server
response = requests.post(url, json=payload)

# Check if the request was successful
if response.status_code == 200:
    # Parse the JSON response
    results = response.json()
    # print(results)
    # If you want to pretty-print the results
    # print(json.dumps(results, indent=2))
else:
    print(f"Request failed with status code {response.status_code}")


# allTFs = response['Integrated--meanRank']


In [2]:
print(results.keys())

dict_keys(['Integrated--meanRank', 'Integrated--topRank', 'GTEx--Coexpression', 'ReMap--ChIP-seq', 'Enrichr--Queries', 'ENCODE--ChIP-seq', 'ARCHS4--Coexpression', 'Literature--ChIP-seq'])


In [9]:
temp = results['Literature--ChIP-seq']
# temp = results['Enrichr--Queries']
modified = []
for res in temp:
    overlap = res.get('Overlapping_Genes','notfound')
    if overlap != '' and overlap != 'notfound':
        modified.append(res)

In [10]:
modified

[{'Query Name': 'myQuery',
  'Rank': '1',
  'Scaled Rank': '0.006098',
  'Set_name': 'HIF1A_21447827_CHIPSEQ_MCF7_HUMAN',
  'TF': 'HIF1A',
  'Intersect': '1',
  'Set length': '292',
  'FET p-value': '0.04255',
  'FDR': '1.0',
  'Odds Ratio': '34.36',
  'Library': 'Literature--ChIP-seq',
  'Overlapping_Genes': 'SCD'},
 {'Query Name': 'myQuery',
  'Rank': '2',
  'Scaled Rank': '0.0122',
  'Set_name': 'NR1H3_23393188_CHIPSEQ_ATHEROSCLEROTICFOAM_HUMAN',
  'TF': 'NR1H3',
  'Intersect': '1',
  'Set length': '577',
  'FET p-value': '0.08179',
  'FDR': '1.0',
  'Odds Ratio': '17.36',
  'Library': 'Literature--ChIP-seq',
  'Overlapping_Genes': 'SCD'},
 {'Query Name': 'myQuery',
  'Rank': '3',
  'Scaled Rank': '0.01829',
  'Set_name': 'NR3C1_23031785_CHIPSEQ_PC12_MOUSE',
  'TF': 'NR3C1',
  'Intersect': '1',
  'Set length': '793',
  'FET p-value': '0.1101',
  'FDR': '1.0',
  'Odds Ratio': '12.63',
  'Library': 'Literature--ChIP-seq',
  'Overlapping_Genes': 'SCD'},
 {'Query Name': 'myQuery',
  'Ra

In [19]:
allTFs = results['Literature--ChIP-seq']
# print(json.dumps(allTFs,indent=2))

for dump in allTFs:
    pmid = dump['Set_name'].split('_')[1]
    human = dump['Set_name'].split('_')[-1]
    if human == "HUMAN":
        overLappingGenes = dump['Overlapping_Genes'].split(',')
        if(targetGene in overLappingGenes):
            new_row = {'Database' : 'CHEA', 'TF' : dump["TF"],'TargetGene':targetGene,'Library/PMID':pmid}
            final_df = pd.concat([final_df, pd.DataFrame([new_row])],ignore_index = True)

In [20]:
final_df

,Database,TF,TargetGene,Library/PMID
0,CHEA,AR,PLA2G5,25329375
1,CHEA,TCF21,PLA2G5,26020271
2,CHEA,MYC,PLA2G5,19915707
3,CHEA,PBX1,PLA2G5,22567123
4,CHEA,STAT3,PLA2G5,23295773
5,CHEA,TP63,PLA2G4B,22573176
6,CHEA,MYCN,CHEK1,21190229
7,CHEA,STAT1,CHEK1,20625510
8,CHEA,E2F4,CHEK1,17652178
9,CHEA,E2F1,CHEK1,21310950


In [21]:
final_df.to_csv('D:/Raylab/LiMeNEx/FetchData/temp1.csv')

trrust

In [19]:
import pandas as pd

df2 = pd.read_csv('D:\\Raylab\\LiMeNEx\\FetchData\\trrust_rawdata.human.tsv',sep = '\t' ,header= None)
df2.columns = ['TF','Target','Regulation','PMID']

df2.head()

df2 = df2[(df2["Target"] == targetGene)]

for i in range(0,len(df2)):
    new_row = {'Database' : 'TRRUST', 'TF' : list(df2['TF'])[i],'TargetGene':targetGene,'Library/PMID':list(df2["PMID"])[i]}
    final_df = pd.concat([final_df, pd.DataFrame([new_row])],ignore_index = True)


In [20]:
final_df

,Database,TF,TargetGene,Library/PMID
0,Signor,PPARD,SLC25A20,19577614
1,Signor,PPARA,SLC25A20,19577614
2,CHEA,HNF4A,SLC25A20,Literature--ChIP-seq
3,CHEA,VDR,SLC25A20,Literature--ChIP-seq
4,CHEA,CDX2,SLC25A20,Literature--ChIP-seq
5,CHEA,PPARD,SLC25A20,Literature--ChIP-seq
6,CHEA,NANOG,SLC25A20,Literature--ChIP-seq
7,CHEA,ERG,SLC25A20,Literature--ChIP-seq
8,CHEA,TCF3,SLC25A20,Literature--ChIP-seq
9,CHEA,SRF,SLC25A20,Literature--ChIP-seq


Save CSV file

In [98]:
final_df.to_csv(f"D:/Raylab/LiMeNEx/FetchData/{targetGene}.csv")